In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from PIL import Image
import glob

# 1. Device and Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1), nn.InstanceNorm2d(channels)
        )
    def forward(self, x): return x + self.conv(x)

class TinyStyleNet(nn.Module):
    def __init__(self):
        super(TinyStyleNet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 9, stride=1, padding=4), nn.InstanceNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.InstanceNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.InstanceNorm2d(128), nn.ReLU(inplace=True)
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(128) for _ in range(5)])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1), nn.InstanceNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.InstanceNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 9, stride=1, padding=4)
        )
    def forward(self, x): return self.decoder(self.res_blocks(self.encoder(x)))

vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()
for param in vgg.parameters(): param.requires_grad_(False)

def get_features(image):
    layers = {'8': 'conv2_2', '10': 'conv3_1', '19': 'conv4_1', '21': 'conv4_2', '28': 'conv5_1'}
    features = {}
    x = image
    for name, layer in vgg._modules.items():
        x = layer(x)
        if name in layers: features[layers[name]] = x
    return features

def gram_matrix(tensor):
    b, c, h, w = tensor.size()
    features = tensor.view(b * c, h * w)
    return torch.mm(features, features.t()).div(b * c * h * w)

def train():
    style_net = TinyStyleNet().to(device)
    optimizer = optim.Adam(style_net.parameters(), lr=0.001)

    content_weight = 1e4   
    style_weight = 1e10    

    style_transform = transforms.Compose([
        transforms.Resize((256, 256)), 
        transforms.ToTensor()
    ])
    
    style_img = Image.open("Styles/StarryNight.jpg").convert('RGB')
    style_tensor = style_transform(style_img).unsqueeze(0).to(device)
    style_features = get_features(style_tensor)
    style_grams = {layer: gram_matrix(style_features[layer]) for layer in style_features}

    train_images = glob.glob("train_images/*.jpg")
    print(f"Starting training on {len(train_images)} images for 2 epochs...")

    for epoch in range(2):
        for i, img_path in enumerate(train_images):
            content_img = Image.open(img_path).convert('RGB')
            content_tensor = style_transform(content_img).unsqueeze(0).to(device)

            generated_tensor = style_net(content_tensor)

            gen_features = get_features(generated_tensor)
            content_features = get_features(content_tensor)

            content_loss = content_weight * torch.mean((gen_features['conv4_2'] - content_features['conv4_2'])**2)
            
            style_loss = 0
            for layer in ['conv2_2', 'conv3_1', 'conv4_1', 'conv5_1']:
                style_loss += style_weight * torch.mean((gram_matrix(gen_features[layer]) - style_grams[layer])**2)
            
            total_loss = content_loss + style_loss

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            if i % 10 == 0:
                print(f"Epoch {epoch} | Image {i}/{len(train_images)} | Loss: {total_loss.item():.2f}")

            if i % 1000 == 0:
                with torch.no_grad():
                    preview = generated_tensor.detach().cpu().squeeze().clamp(0, 1).numpy().transpose(1, 2, 0)
                    preview_img = Image.fromarray((preview * 255).astype('uint8'))
                    preview_img.save(f"Preview/training_peek_epoch{epoch}_{i}.jpg") 
                    print(f"New preview saved: training_peek_epoch{epoch}_{i}.jpg")

    torch.save(style_net.state_dict(), "starry_night_model.pth")
    print("Training Complete! Saved as starry_night_model.pth")

if __name__ == "__main__":
    train()

Starting training on 40504 images for 2 epochs...
Epoch 0 | Image 0/40504 | Loss: 3332836.25
New preview saved: training_peek_epoch0_0.jpg
Epoch 0 | Image 10/40504 | Loss: 2204891.50
Epoch 0 | Image 20/40504 | Loss: 999338.31
Epoch 0 | Image 30/40504 | Loss: 541634.44
Epoch 0 | Image 40/40504 | Loss: 375378.62
Epoch 0 | Image 50/40504 | Loss: 425528.34
Epoch 0 | Image 60/40504 | Loss: 289170.97
Epoch 0 | Image 70/40504 | Loss: 308725.59
Epoch 0 | Image 80/40504 | Loss: 305071.34
Epoch 0 | Image 90/40504 | Loss: 297197.81
Epoch 0 | Image 100/40504 | Loss: 266538.44
Epoch 0 | Image 110/40504 | Loss: 239107.41
Epoch 0 | Image 120/40504 | Loss: 254067.27
Epoch 0 | Image 130/40504 | Loss: 249728.17
Epoch 0 | Image 140/40504 | Loss: 268564.88
Epoch 0 | Image 150/40504 | Loss: 238452.28
Epoch 0 | Image 160/40504 | Loss: 267667.25
Epoch 0 | Image 170/40504 | Loss: 229998.67
Epoch 0 | Image 180/40504 | Loss: 235312.11
Epoch 0 | Image 190/40504 | Loss: 263713.84
Epoch 0 | Image 200/40504 | Loss: